In [ ]:
# Import required packages.
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
import matplotlib.image as mpimg
import geopandas as gpd
from shapely.geometry import Point
from shapely.geometry import Polygon
import statsmodels.api as sm
import statsmodels.formula.api as smf
import folium
from folium.plugins import HeatMap

df_b = pd.read_csv("joined_glm.csv") # Read in the decoded behaviour states data.
gpipff = pd.read_csv("gpipff4.csv") # Read in pre-calculated paddock-agnostic metrics data.

df_b # Display top 5 rows and bottom 5 rows of the dataframe.

In [ ]:
# Convert to a GeoDataFrame (Set EPSG code).
geometry = [Point(xy) for xy in zip(df_b['x'], df_b['y'])]
points_gdf = gpd.GeoDataFrame(df_b, geometry=geometry, crs="EPSG:27700")

# Export to a GeoPackage (.gpkg) file for QGIS.
points_gdf.to_file("qgis_cattle.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

In [ ]:
# Seperate grazing states from resting and transit states.
g_df = df_b[df_b["behaviour_state"].isin(["Intensive Foraging", "Extensive Foraging"])]

# Convert to a GeoDataFrame (Set EPSG code).
geometry = [Point(xy) for xy in zip(g_df['x'], g_df['y'])]
points_gdf = gpd.GeoDataFrame(g_df, geometry=geometry, crs="EPSG:27700")

# Export to a GeoPackage (.gpkg) file for QGIS.
points_gdf.to_file("qgis_g.gpkg", layer="telemetry_fixes", driver="GPKG")
print("Spatial file successfully created for QGIS!")

In [ ]:
# Inspect the final hmm raw states before naming them to ensure data is correct.
state_summary = ( 
    df_b.groupby("assigned_state") 
    .agg( 
        mean_step_m=("step_length_m", "mean"), 
        median_step_m=("step_length_m", "median"), 
        mean_speed=("speed_m_per_min", "mean"), 
        mean_abs_turn=("abs_turning_angle", "mean"), 
        n=("assigned_state", "size") 
    )
    .reset_index() 
) 

print(state_summary) 

In [ ]:
# Label raw state numbers (only after looking at the state summary).
state_labels = { 
    1: "Resting",
    2: "Intensive Foraging",
    3: "Extensive Foraging",
    4: "Transit",
}

df_b["behaviour_state"] = df_b["assigned_state"].map(state_labels)

In [ ]:
# Behaviour state speed distribution plot.
custom_colors = {
    "Resting": "grey",
    "Intensive Foraging": "lightgreen",
    "Extensive Foraging": "darkgreen",
    "Transit": "orange"
}

fig, ax = plt.subplots(figsize=(5, 4), dpi=300)

sns.kdeplot(
    data=df_b, x="speed_m_per_min", hue="behaviour_state", 
    fill=True, common_norm=False, 
    palette=custom_colors,   
    alpha=0.35,
    linewidth=1.2,           
    cut=0,
    ax=ax
)

ax.set_xlim(left=0) # Enforce boundary (Speed cannot drop below 0).

ax.set_title("Speed Distribution by Behaviour State", fontsize=11, fontweight='bold', pad=10, loc='left')
ax.set_xlabel(r"Speed (m min$^{-1}$)", fontsize=10, labelpad=6)
ax.set_ylabel("Density", fontsize=10, labelpad=6)

ax.tick_params(labelsize=9)
sns.despine(trim=True)
plt.tight_layout()

plt.savefig("velocity_b_kde.png", dpi=300, bbox_inches='tight') # Save figure.

plt.show()

In [ ]:
# Behaviour state turning angle distribution plot.
custom_colors = {
    "Resting": "grey",
    "Intensive Foraging": "lightgreen",
    "Extensive Foraging": "darkgreen",
    "Transit": "orange"
}

fig, ax = plt.subplots(figsize=(5, 4), dpi=300)

sns.kdeplot(
    data=df_b, x="turning_angle", hue="behaviour_state", 
    fill=True, common_norm=False, 
    palette=custom_colors,
    alpha=0.4,
    linewidth=1.5,
    ax=ax
)

ax.set_title("Empirical Turning Angles by Behaviour State", fontsize=11, fontweight='bold', pad=10, loc='left')
ax.set_xlabel(r"Turning Angle ($\alpha$, rad)", fontsize=10, labelpad=6)
ax.set_ylabel("Density", fontsize=10, labelpad=6)

ax.tick_params(labelsize=9)
sns.despine(trim=True)
plt.tight_layout()

plt.savefig("degree_b_kde.png", dpi=300, bbox_inches='tight') # Save figure.

plt.show()

In [ ]:
# Folium interactive heatmap.
geo_data = df_b[['Latitude', 'Longitude', 'Position_Time']].dropna()

heat_data = geo_data[['Latitude', 'Longitude']].values.tolist()

map_center = [geo_data['Latitude'].mean(), geo_data['Longitude'].mean()]
m = folium.Map(location=map_center, zoom_start=16)

HeatMap(heat_data, radius=1.5, blur=0.3, max_zoom=1).add_to(m)

m

In [ ]:
# Calculate time allocations for behaviour states by NVC community.
profile = ( 
    df_b.groupby(["NVC", "behaviour_state"])
    .size() 
    .groupby(level=0) 
    .apply(lambda x: 100 * x / x.sum()) 
) 

# Remove the names from the multi-index to prevent name clashing.
profile.index.names = [None] * len(profile.index.names)

# Reset the index now the names are removed.
df_clean1 = profile.reset_index(name="Allocation")

# Overwrite the column names based on their positions.
df_clean1.columns = ['Serial', 'Extra_Serial', 'behaviour_state', 'Allocation']

# Drop the duplicate column.
df_clean1 = df_clean1.drop(columns=['Extra_Serial'])

df_clean1

In [ ]:
# Behaviour time allocation figure.
# Pivot data and enforce a logical behavioral sequence.
state_order = ["Resting", "Intensive Foraging", "Extensive Foraging", "Transit"]
df_pivot = df_clean1.pivot(index="Serial", columns="behaviour_state", values="Allocation")[state_order]

# Map behavioral colours.
custom_colors = {
    "Resting": "grey",            
    "Intensive Foraging": "lightgreen",   
    "Extensive Foraging": "darkgreen",   
    "Transit": "orange"                          
}

# Initialise figure canvas.
fig, ax = plt.subplots(figsize=(6, 8), dpi=300)

# Plot horizontal stacked bars.
df_pivot.plot(
    kind="barh", stacked=True, ax=ax, width=0.6,
    color=[custom_colors[s] for s in state_order],
    edgecolor="white", linewidth=0.5
)

# Formatting and axis cleanup.
ax.set_title("Habitat Behaviour Time Budget Allocations", fontsize=11, fontweight="bold", pad=12, loc="left")
ax.set_ylabel("Habitat Classification", fontsize=10, labelpad=6)
ax.set_xlabel("Time Budget Allocation (%)", fontsize=10, labelpad=6)

ax.set_xlim(0, 100)  # Enforce percentage boundary limit.
ax.set_yticklabels(df_pivot.index, fontsize=8)  
ax.tick_params(labelsize=9)

# Clean background grid lines.
ax.grid(axis='x', linestyle=':', alpha=0.4, color='#ced4da', zorder=0)
ax.set_axisbelow(True)

# Remove heavy top and right frame outlines.
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

# Position the legend box on the right hand side.
ax.legend(
    loc="upper left", bbox_to_anchor=(1.02, 1.0),
    frameon=True, facecolor="white", edgecolor="#e9ecef", fontsize=9,
    title="Behaviour State", title_fontsize=9
)

plt.tight_layout()

# Save the figure with a tight bounding box to prevent legend clipping.
plt.savefig('habitat_budgets_b.png', dpi=300, bbox_inches='tight')

# Show figure.
plt.show()

In [ ]:
# Try importing adjustText to auto-resolve label overlaps. 
try:
    from adjustText import adjust_text
except ImportError:
    adjust_text = None

# Set style.
plt.rcParams['xtick.labelsize'] = 9
plt.rcParams['ytick.labelsize'] = 9
plt.rcParams['axes.linewidth'] = 0.8

# Layout specifications.
fig = plt.figure(figsize=(11, 5.5), dpi=300)

# Data initialisation.
habitat_summary = {
    'Habitat': gpipff["NVC"],
    'Expected Grazing Count Per Day': gpipff["Exp_p_day"],
    'PFF': gpipff["PFF"]
}
df_plot = pd.DataFrame(habitat_summary)

# Create the numeric IDs.
df_plot['Habitat_ID'] = range(1, len(df_plot) + 1)

# Position the main plot axis explicitly to prevent collisions with the table.
ax = fig.add_axes([0.08, 0.12, 0.48, 0.75])

# Draw quadrant background zones based on pre-calculated thresholds (mean PFF).
threshold_x = 4.330000
threshold_y = 0.059435

ax.axvspan(0.0, threshold_x, 0.11887, 1.0, color='#495057', alpha=0.03, zorder=0)  # Top Left
ax.axvspan(threshold_x, 12, 0.11887, 1.0, color='#c92a2a', alpha=0.03, zorder=0)  # Top Right

# Draw threshold lines.
ax.axvline(x=threshold_x, color='#6c757d', linestyle='--', linewidth=0.8, zorder=1)
ax.axhline(y=threshold_y, color='#6c757d', linestyle='--', linewidth=0.8, zorder=1)

# Plot data points.
sns.scatterplot(
    data=df_plot, x='Expected Grazing Count Per Day', y='PFF', 
    s=0, color='#005a9c', edgecolor='white', linewidth=0.6, zorder=4, ax=ax
)

# Quadrant Text (Aligned to axes fractions).
ax.text(0.03, 0.96, 'High Risk / Avoided', transform=ax.transAxes, 
        fontsize=8.5, color='#495057', fontweight='bold', va='top')
ax.text(0.97, 0.96, 'High Risk / Preferred\n(Vulnerable)', transform=ax.transAxes, 
        fontsize=8.5, color='#c92a2a', fontweight='bold', va='top', ha='right')
ax.text(0.03, 0.04, 'Low Risk / Avoided', transform=ax.transAxes, 
        fontsize=8.5, color='#6c757d', va='bottom')
ax.text(0.97, 0.04, 'Low Risk / Preferred', transform=ax.transAxes, 
        fontsize=8.5, color='#6c757d', va='bottom', ha='right')

# Numeric labels.
texts = []
for i, row in df_plot.iterrows():
    t = ax.text(row['Expected Grazing Count Per Day'], row['PFF'], str(row['Habitat_ID']), 
                fontsize=7, fontweight='bold', va='center', ha='center', zorder=5)
    texts.append(t)

if adjust_text:
    adjust_text(
        texts, 
        arrowprops=dict(arrowstyle="-", color="#adb5bd", linewidth=0.5),
        force_points=(0.2, 0.4), 
        expand_points=(1.3, 1.3),
        ax=ax
    )

# Axes limits.
ax.set_xlim(3.25, 5.25)
ax.set_ylim(0.0, 0.5)
ax.grid(True, linestyle=':', alpha=0.3, color='#ced4da', zorder=2)

# Labels and cleanup.
ax.set_xlabel('Expected Grazing Count Per Day', fontsize=10, labelpad=6)
ax.set_ylabel('Proportional Foraging Footprint ($PFF_H$)', fontsize=10, labelpad=6)
ax.set_title('Habitat Spatial Vulnerability Matrix', fontsize=11, fontweight='bold', pad=12, loc='left')

# Split and render the NVC communities into a clean 2-column legend table.
legend_data = list(zip(df_plot['Habitat_ID'], df_plot['Habitat']))
midpoint = (len(legend_data) + 1) // 2
col1_data = legend_data[:midpoint]
col2_data = legend_data[midpoint:]

table_rows = []
for i in range(midpoint):
    id1, name1 = col1_data[i]
    id2, name2 = col2_data[i] if i < len(col2_data) else ("", "")
    table_rows.append([id1, name1, id2, name2])

# Place table to the right side of the main plot.
table_ax = fig.add_axes([0.60, 0.12, 0.36, 0.75]) 
table_ax.axis('off')

matrix_table = table_ax.table(
    cellText=table_rows,
    colLabels=['No.', 'Habitat (NVC)', 'No.', 'Habitat (NVC)'],
    colWidths=[0.07, 0.43, 0.07, 0.43],
    loc='center',
    cellLoc='left'
)

# Apply font scaling and remove interior lines.
matrix_table.auto_set_font_size(False)
matrix_table.set_fontsize(7.5)
for (row_idx, col_idx), cell in matrix_table.get_celld().items():
    cell.set_linewidth(0.4)
    cell.set_edgecolor('#dee2e6')
    # Bold headers and index columns
    if row_idx == 0:
        cell.set_text_props(fontweight='bold')
        cell.set_facecolor('#f8f9fa')
    elif col_idx == 0 or col_idx == 2:
        cell.set_text_props(fontweight='bold')

sns.despine(ax=ax, trim=True) 

# Save output.
plt.savefig('Figure_2_Vulnerability_Matrix.png', bbox_inches='tight', dpi=300)
plt.show()